# Firely v1.2: armonización y validación sobre un recorte de Coahuila

Este notebook ejecuta **el mismo código** de `pipeline.harmonize` sobre puntos FIRMS ya procesados. El recorte es un rectángulo de ejemplo, no el límite oficial de Coahuila; sus métricas no sustituyen la validación por país de `docs/metodologia-armonizacion.md`. Incluye los años 2001–2025 para poder ocultar un año de traslape a la vez. No usa llaves ni descarga datos.

Desde la raíz del repositorio, instala las dependencias del pipeline en `.venv` y ejecuta `jupyter lab notebooks/armonizacion-coahuila.ipynb`. Se necesita `data/processed/hotspots/Mexico/year=*/part-0.parquet`, generado con `pipeline.ingest` y `pipeline.h3_cells`, y opcionalmente `cell_biome/Mexico.parquet`. El recorte ocupa espacio temporal y la validación puede tardar varios minutos.


In [ ]:
from pathlib import Path
import json, tempfile
import duckdb
import pandas as pd
from pipeline import harmonize

PROCESSED = Path("data/processed").resolve()
HOTSPOTS = PROCESSED / "hotspots" / "Mexico"
assert (HOTSPOTS / "year=2001" / "part-0.parquet").exists(), "Genera primero los hotspots procesados de México"
BOX = (-103.9, 24.5, -99.6, 29.9)  # oeste, sur, este, norte; ejemplo
work = Path(tempfile.mkdtemp(prefix="firely-harm-example-"))
example = work / "processed"
(example / "metadata").mkdir(parents=True)
print("Área de ejemplo:", BOX, "Archivos temporales:", work)


## 1. Preparar el mismo esquema de entrada que usa el pipeline

Se copian solo las detecciones dentro del rectángulo y los biomas de las celdas presentes. La entrada queda particionada por año, igual que el procesamiento normal.


In [ ]:
con = duckdb.connect()
for year in range(2001, 2026):
    source = HOTSPOTS / f"year={year}" / "part-0.parquet"
    if not source.exists():
        raise FileNotFoundError(source)
    target = example / "hotspots" / "Mexico" / f"year={year}" / "part-0.parquet"
    target.parent.mkdir(parents=True, exist_ok=True)
    con.execute(f"COPY (SELECT * FROM read_parquet({harmonize.q(source)}) WHERE lon BETWEEN {BOX[0]} AND {BOX[2]} AND lat BETWEEN {BOX[1]} AND {BOX[3]}) TO {harmonize.q(target)} (FORMAT PARQUET, COMPRESSION ZSTD)")
biome = PROCESSED / "cell_biome" / "Mexico.parquet"
if biome.exists():
    target = example / "cell_biome" / "Mexico.parquet"
    target.parent.mkdir(parents=True, exist_ok=True)
    con.execute(f"COPY (SELECT b.* FROM read_parquet({harmonize.q(biome)}) b SEMI JOIN read_parquet({harmonize.q(example / 'hotspots' / 'Mexico' / 'year=*' / '*.parquet')}) h USING (h3_r6)) TO {harmonize.q(target)} (FORMAT PARQUET)")
print("Detecciones del recorte:", con.execute("SELECT count(*) FROM read_parquet(?)", [str(example / 'hotspots' / 'Mexico' / 'year=*' / '*.parquet')]).fetchone()[0])
con.close()


## 2. Ejecutar v1.2 y su validación año-fuera

`harmonize_country` construye la matriz celda × mes, ajusta razones mensuales por agrupación, reparte a celdas sin detecciones MODIS, evalúa 2012–2025 dejando fuera un año cada vez y selecciona la configuración de menor error anual. Los CSV de cada pliegue quedan en `metadata/`.


In [ ]:
con = duckdb.connect()
con.execute("SET memory_limit='3GB'")
meta = harmonize.harmonize_country(con, example, "Mexico", example)
con.close()
print("Versión:", harmonize.METHODOLOGY_VERSION)
print("Modelo elegido:", meta["model"])
print("Celdas:", meta["n_cells"])
pd.DataFrame({name: {"error_anual_pct": scores["ta"]["region_year_mape_pct"],
                     "factor_fijo_pct": scores["ta"]["region_year_mape_baseline_pct"],
                     "spearman_mensual": scores["ta"]["region_month_spearman"]}
              for name, scores in meta["validation_by_model"].items()}).T


## 3. Inspeccionar la validación y la serie armonizada

El factor fijo es el comparador. Los años 2001–2011 son estimados y deben mostrarse como tales.


In [ ]:
chosen = meta["model"]
folds = pd.read_csv(example / "metadata" / f"loyo_region_month_Mexico_{chosen}_ta.csv")
annual = folds.groupby("year")[["actual", "est", "base"]].sum()
annual["error_metodo_pct"] = 100 * (annual["est"] - annual["actual"]).abs() / annual["actual"].replace(0, float("nan"))
annual["error_factor_fijo_pct"] = 100 * (annual["base"] - annual["actual"]).abs() / annual["actual"].replace(0, float("nan"))
display(annual.round(2))
print("Error anual medio del recorte:", round(annual["error_metodo_pct"].mean(), 2), "%")


In [ ]:
series = con = duckdb.connect()
series = con.execute("SELECT year, month, sum(count_harm) AS snpp_equivalentes, bool_or(is_estimated) AS estimado FROM read_parquet(?) GROUP BY 1,2 ORDER BY 1,2", [str(example / "cell_month_harmonized" / "Mexico" / "year=*" / "*.parquet")]).df()
con.close()
display(series.head(), series.tail())


## 4. Reproducibilidad y publicación

Los CSV `loyo_*`, Parquet `cell_month_harmonized` y `harmonization_summary.json` permiten auditar el cálculo. Para el dataset público se deben incluir versión del código, corte de datos, cita de NASA FIRMS y licencia de los derivados. Este ejemplo evita publicar datos fuente de terceros sin revisar sus términos.
